# N=96 layer ladder: L\*(96) = 8, and L\*(T) rises by one rung

**Job 49639312**, `sud2n96L80k`, COMPLETED, elapsed 17:45:48, ExitCode 0:0.

**Headline.** At N=96 with `n_train=80000`, the 0.90 gate at rho=0.60 is cleared in **0 of 6** depth
bins at L=4, **3 of 6** at L=6, and **6 of 6** at L=8. So L\*(96) = 8, and the N ladder now reads
L\* = 3, 4, 8 at N = 24, 48, 96 (all at ff=1024). The three bins L=6 misses are the three *deepest*
(T=7, 8, 9), so L\*(T) at N=96 is 6 for T<=6 and 8 for T>=7 -- the first resolvable rise of L\* with
reasoning depth in this project, against finding 11's flat +0.00 layers per round over 20 bins.

**This is a leaked-order run** (`shuffle_adj=False`), so every `solve_rate` below is a *decoding*
score, not a deduction score (findings 16-17). See the caveats at the end.

## Provenance

| item | value |
| --- | --- |
| Slurm job | 49639312 (`sud2n96L80k`), submitted 2026-10-26T10:31:40, ended 2026-10-27T04:18:19 |
| sbatch | `scripts/slurm/sudoku2_n96_layer_scan_80k.sbatch` |
| git commit at submit | `9b98d70` |
| seed | 0 (training); eval pool `seed_start=5e10`, instance *i* at `5e10 + i` |
| config | N=96, degree=5, n_colors=4, hint_mode=removal, `shuffle_adj=False` |
| model | d_model=256, nhead=4, ff=1024, pre-norm, lr=1e-3 **constant**, batch 32 |
| data | n_train=80000, n_val=300, 150 epochs |
| cells | num_layers in {6, 4, 8}, in that order |
| checkpoints | `$SCRATCH/spin_gnn/sudoku_n96_layer_scan_2026-10-26/d0256_h04_L{04,06,08}_ff01024_lr1e-03_pre` |
| reference (previous best) | L=12 cell B of job 49565433, `$SCRATCH/spin_gnn/sudoku_n96_plateau_2026-10-26/d0256_h04_L12_ff01024_lr1e-03_pre` |
| final checkpoint selection | last epoch, not best-val (a known -0.0280 cost, see caveats) |

One change from the reference: `num_layers`. Everything else -- data budget, schedule, width,
seed, generator -- is identical, which is what makes the shared-pool deltas below readable as a
depth effect.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

SCRATCH = os.environ["SCRATCH"]
RUN = f"{SCRATCH}/spin_gnn/sudoku_n96_layer_scan_2026-10-26"
REF = f"{SCRATCH}/spin_gnn/sudoku_n96_plateau_2026-10-26"
GATE = 0.90
LADDER = [4, 6, 8]
COLORS = {4: "#d62728", 6: "#ff7f0e", 8: "#2ca02c", 12: "#1f77b4"}


def with_L(df):
    """Pull num_layers out of the checkpoint name and add the success count k."""
    df = df.copy()
    df["L"] = df.checkpoint.str.extract(r"_L(\d+)_").astype(int)
    if "n" in df.columns:
        df["k"] = (df.solve_rate * df.n).round().astype(int)
    return df


depth = with_L(pd.read_csv(f"{RUN}/depth_by_layers_n96.csv"))
pool = with_L(pd.read_csv(f"{RUN}/sharedpool_n1500.csv")).set_index("L").sort_index()
sweep = pd.read_csv(f"{RUN}/sweep.csv").set_index("num_layers").sort_index()

wide = depth[depth.rho_requested == 0.60].copy()      # the measurement
shallow = depth[depth.rho_requested == 0.75].copy()   # the control
print(f"cells: {sorted(sweep.index)}   rho=0.60 bins: {sorted(wide['T'].unique())}")
print(f"realized density at rho=0.60: {sorted(wide.realized_rho.unique())}")

## 1. The metric of record: the shared pool

n=1500 instances drawn from the shared stream, every cell scored on the *same* instances, so the
comparison against the reference is paired (McNemar exact on the discordant pairs).

In [ ]:
cols = ["n_instances", "solve_rate", "wilson_lo", "wilson_hi", "delta",
        "discordant_won", "discordant_lost", "mcnemar_exact_p"]
tbl = pool[cols].copy()
tbl["n_params"] = [sweep.loc[L, "n_params"] if L in sweep.index else np.nan for L in tbl.index]
tbl["final_val_nll"] = [sweep.loc[L, "final_val_nll"] if L in sweep.index else np.nan
                        for L in tbl.index]
print("shared pool, n=1500, paired vs L=12 (the reference row has delta 0 by construction)")
print(tbl.to_string(float_format=lambda v: f"{v:.4f}"))

Read: **L=8 ties the reference** (0.9113 vs 0.9273, delta -0.0160, p=0.091 -- not resolvable either
way) at 1.5x fewer layers, while L=6 (-0.0440, p=1.1e-05) and L=4 (-0.1673, p=1.7e-42) are clearly
worse. The aggregate alone cannot locate L\*, because the pool mixes depths; that is what section 2
is for.

## 2. Depth-resolved: where the gate is cleared

L\* is defined as the smallest `num_layers` that reaches 0.90 on instances of DD depth T **at fixed
density**. Fixing density matters: deeper instances are also sparser unless you hold rho, and
sparsity alone makes them harder. The realized density is identical (0.594) in all six bins.

In [ ]:
def bin_table(df, label):
    Ts = sorted(df["T"].unique())
    rows = {}
    for L in sorted(df.L.unique()):
        g = df[df.L == L].set_index("T")
        rows[f"L={L}"] = [g.loc[t, "solve_rate"] for t in Ts]
    out = pd.DataFrame(rows, index=[f"T={t}" for t in Ts]).T
    out["cleared"] = [f"{int((out.loc[i] >= GATE).sum())}/{len(Ts)}" for i in out.index]
    # NB `df.T` is DataFrame.transpose; the depth column always needs brackets.
    per_bin = df.groupby("T").n.first().astype(int)
    print(f"{label}  (n per bin: {dict(per_bin)}, total {int(per_bin.sum())} per row)")
    print(out.to_string(float_format=lambda v: f"{v:.3f}"))
    return out


_ = bin_table(wide, "rho=0.60, THE MEASUREMENT")
print()
_ = bin_table(shallow, "rho=0.75, the shallow control")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, df, title in ((axes[0], wide, "rho=0.60 (the measurement)"),
                      (axes[1], shallow, "rho=0.75 (shallow control)")):
    for L in sorted(df.L.unique()):
        g = df[df.L == L].sort_values("T")
        lo, hi = [], []
        for _, r in g.iterrows():
            a, b = stats.beta.interval(0.95, r.k + 0.5, r.n - r.k + 0.5)  # Jeffreys
            lo.append(r.solve_rate - a)
            hi.append(b - r.solve_rate)
        lo, hi = np.clip(lo, 0, None), np.clip(hi, 0, None)
        ax.errorbar(g["T"], g.solve_rate, yerr=[lo, hi], fmt="-o", color=COLORS[L],
                    ms=4, lw=1.8, capsize=2, label=f"L={L}")
    ax.axhline(GATE, color="0.35", ls="--", lw=1.2)
    ax.set_xlabel("DD depth T")
    ax.set_ylabel("solve_rate")
    ax.set_title(title)
    ax.set_xticks(sorted(df["T"].unique()))
    ax.set_ylim(0.6, 1.01)
    ax.legend(loc="lower left", fontsize=9)
fig.suptitle("N=96 layer ladder: only L=6 sits on the gate, and only L=6 has a depth trend")
fig.tight_layout()
plt.show()

The control is the reason the rho=0.60 panel is readable as depth rather than difficulty: at
rho=0.75 every cell clears every bin (L=8 is 1.000 in all four), so L=4 and L=6 are not broken
models -- they fail only where the instances are deep.

## 3. Is the L=6 decline a trend, or six noisy bins?

Two tests, both pre-specified in shape by finding 11, which reported its own weak signal at
p=0.088 using the first of them:

- **Cochran-Armitage** trend in solve_rate across the ordered depth bins.
- **Pooled shallow (T<=6) vs deep (T>=7)**, Fisher exact.

They are *not independent* -- they look at the same six bins -- so two marginal p-values are one
marginal result, not two.

In [ ]:
def ca_trend(T, k, n):
    """Cochran-Armitage trend across ordered depth bins (finding 11's statistic)."""
    T, k, n = np.asarray(T, float), np.asarray(k, float), np.asarray(n, float)
    N, K = n.sum(), k.sum()
    p = K / N
    Tbar = (n * T).sum() / N
    z = (k * (T - Tbar)).sum() / np.sqrt(p * (1 - p) * (n * (T - Tbar) ** 2).sum())
    return z, 2 * stats.norm.sf(abs(z))


def split_test(g, cut=6):
    """Pooled shallow (T<=cut) vs deep (T>cut) solve_rate, Fisher exact."""
    sh, dp = g[g["T"] <= cut], g[g["T"] > cut]
    a, b = int(sh.k.sum()), int(sh.n.sum() - sh.k.sum())
    c, e = int(dp.k.sum()), int(dp.n.sum() - dp.k.sum())
    return a / (a + b), c / (c + e), stats.fisher_exact([[a, b], [c, e]])[1]


rows = []
for L in LADDER:
    g = wide[wide.L == L].sort_values("T")
    z, p = ca_trend(g["T"], g.k, g.n)
    r_sh, r_dp, p_f = split_test(g)
    rows.append({"L": L, "pooled": g.k.sum() / g.n.sum(), "CA z": z, "CA p": p,
                 "T<=6": r_sh, "T>=7": r_dp, "delta": r_sh - r_dp, "Fisher p": p_f,
                 "cleared": f"{int((g.solve_rate >= GATE).sum())}/6"})
print(pd.DataFrame(rows).set_index("L").to_string(float_format=lambda v: f"{v:.4f}"))

**The pattern is the evidence, more than either p-value is.** L=4 is on the floor (0.753) and L=8 is
above the gate everywhere (0.937), so neither *can* show a depth trend, and neither does (p=0.42,
p=0.81). L=6 sits at 0.897, within a hair of the gate, and it is the one cell with a trend
(CA p=0.055) and a shallow-vs-deep gap (+0.054, Fisher p=0.072). A depth cost that appears only at
the layer count near the threshold, and at neither of the two that are pinned, is what a real
threshold effect looks like; it is not what flatness predicts.

This is why finding 11's flat result is demoted rather than contradicted: at N=24, T spans 1-5 and
every cell is floored or saturated, so that measurement had no cell where the effect could show.

## 4. Comparison to the previous best, on the same eval

The previous best at N=96 is **L=12 cell B of job 49565433** (0.9273 on this same shared pool), the
cell that opened the gate for this scan. Both the shared pool and the depth bins below are the same
eval for both runs: same `seed_start`, same instance ids, same rho, same n per bin.

In [ ]:
ref = with_L(pd.read_csv(f"{REF}/depth_n96_plateau.csv"))
ref = ref[(ref.rho_requested == 0.60) & (~ref.checkpoint.str.endswith("_cos"))]
Ts = sorted(wide["T"].unique())
assert sorted(ref["T"].unique()) == Ts, "depth bins differ; the comparison would not be paired"
assert (wide.groupby("T").n.first() == ref.groupby("T").n.first()).all(), "n per bin differs"

cmp = pd.DataFrame(
    {f"L={L}": [g.set_index("T").loc[t, "solve_rate"] for t in Ts]
     for L, g in list(wide.groupby("L")) + [(12, ref)]},
    index=[f"T={t}" for t in Ts]).T
cmp["cleared"] = [(cmp.loc[i, [f"T={t}" for t in Ts]] >= GATE).sum() for i in cmp.index]
cmp["pool"] = [pool.loc[int(i.split("=")[1]), "solve_rate"] for i in cmp.index]
print("solve_rate by depth bin at rho=0.60, this scan vs the L=12 reference")
print(cmp.to_string(float_format=lambda v: f"{v:.3f}"))
print("\nL=8 vs L=12 on the shared pool: "
      f"{pool.loc[8, 'solve_rate']:.4f} vs {pool.loc[12, 'solve_rate']:.4f}, "
      f"delta {pool.loc[8, 'delta']:+.4f}, McNemar exact p={pool.loc[8, 'mcnemar_exact_p']:.3f} "
      f"({int(pool.loc[8, 'discordant_won'])} won / {int(pool.loc[8, 'discordant_lost'])} lost)")

Both L=8 and L=12 clear all six bins; L=8 does it with 6.58M parameters against 9.74M, and the two
are tied on the pool. So the honest statement is **L\*(96) is 8 as measured on this grid, and the
true threshold lies in (6, 8]** -- L=10 was not run, and would narrow it to one rung.

## 5. Pre-registrations, as resolved

Written into `scripts/slurm/sudoku2_n96_layer_scan_80k.sbatch` before the job ran.

| | claim | verdict | evidence |
| --- | --- | --- | --- |
| (a) | L\*(96) >= 6 | **CONFIRMED** | L=4 clears 0 of 6 bins; L\* = 8 |
| (b) | L\*(96) is DEFINED inside {4, 6, 8} | **CONFIRMED** | L=8 clears 6 of 6 at rho=0.60 |
| (c) | L\*(T) is flat in T, as in finding 11 | **FALSIFIED** (marginally) | L=6 clears T=4,5,6 and fails T=7,8,9; CA p=0.055, shallow-vs-deep 0.861 vs 0.915, Fisher p=0.072. Only at L=6. |
| (d) | L=8 >= L=12, separating the threshold from the optimum | **TIE** | 0.9113 vs 0.9273, delta -0.0160, p=0.091 -- not resolvable either way |

(c) is the one worth being careful about: it is the prediction I most wanted to be wrong, and a
result that agrees with the prediction you wanted to fail deserves more scepticism than one that
does not. I report it as falsified rather than unresolved because of the pattern across cells
(section 3), not because 0.055 crossed a line.

## 6. What this changes

- L\*(96) was **undefined** after finding 15 (the control stalled at 0.8267 and cleared no bin). It
  is now 8, so the N ladder reads L\* = 3, 4, 8 at N = 24, 48, 96.
- Finding 11's flat L\*(T) is demoted from "depth does not track depth" to "not resolvable at
  N=24". At N=96, with a cell on the gate and T spanning 4-9, it rises by one rung.
- The N=96 question is no longer "can it learn at all" but "how deep must it be".

## Caveats that apply

- **Leaked order.** `shuffle_adj=False`, so every `solve_rate` here is a decoding score. Finding 16
  showed this exact family of checkpoints collapses to 0.03-0.15 when `adj[v]` is permuted, so L\*
  measured on the shortcut task need not equal L\* on the real one. A leak-free N=96 run has not
  been done.
- **One seed, two dependent marginal tests.** p=0.055 and p=0.072, seed 0 only.
- **L\* is defined at fixed width.** Finding 14 showed L\* depends on `dim_feedforward`; every L\*
  quoted here is "at ff=1024". A thin-deep N=96 cell (ff=256) could move it and has not been run.
- **The data-budget confound.** L\*(24)=3 and L\*(48)=4 were measured at n_train=20000; this arm is
  at 80000, because nothing at 20000 clears the gate at N=96. The companion N=48 data control
  closes part of that hole; the three numbers are not a clean trend on their own.
- **Last-epoch checkpoints**, not best-val (a measured -0.0280, p=0.0016 cost), applied identically
  to every cell including the reference.
- **The grid, not the threshold.** L=10 was not run, so L\*(96)=8 means "8 on {4,6,8,12}"; the true
  threshold is in (6, 8].

## What I would do next

1. The leak-free L\*(T) ladder, but at **N=48** rather than only N=24 -- this run is the argument
   for it. N=24's T range is 2-5, the range that produced the flat answer.
2. **L=10 at N=96** would tighten L\* from "between 6 and 8" to a single rung, and is one cell.
3. A **thin-deep N=96 cell** (ff=256), which finding 13 says should match L=8/ff=1024 at roughly a
   third of the parameters.